# 대피도 인식 모델 학습 — 완성형 `ml` 폴더용

이 노트북은 함께 제공된 **완성형 `ml` 폴더 ZIP 하나만 업로드**하면 실행되도록 구성되어 있습니다.

기본 구성:

- 합성 YOLO 정답 데이터 **500장**: train 345 / val 95 / test 60
- 공개 실제 대피도 촬영환경 이미지 **208장**: 정답 라벨은 없으며 밑라벨 생성 후 수동 검수 필요
- 클래스 8개: `exit`, `stair`, `elevator`, `extinguisher`, `hydrant`, `you_are_here`, `door`, `room`
- 실제 원본 그룹 단위 분할로 동일 원본 변형이 train/val 양쪽에 들어가는 누수 방지

> 모델 출력과 자동 라벨은 안전 판단의 초안입니다. 실제 대피 안내에 사용하기 전, 모든 객체와 경로를 사람이 검수해야 합니다.

먼저 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤 위에서부터 실행하세요.


In [ ]:
# ============================================================
# Google Drive 체크포인트에서 ml 작업 상태 복원
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import shutil
import os
import sys

DRIVE_DIR = Path(
    "/content/drive/MyDrive/evacuation_checkpoints"
)

checkpoints = sorted(
    DRIVE_DIR.glob("evac_ml_checkpoint_*.zip"),
    key=lambda path: path.stat().st_mtime,
)

assert checkpoints, (
    f"{DRIVE_DIR}에서 체크포인트 ZIP을 찾지 못했습니다."
)

# 가장 최근 체크포인트 선택
CHECKPOINT = checkpoints[-1]

print("복원할 체크포인트:", CHECKPOINT)
print(
    "파일 크기:",
    round(CHECKPOINT.stat().st_size / 1024 / 1024, 2),
    "MB",
)

WORK = Path("/content/evac_work")

shutil.rmtree(
    WORK,
    ignore_errors=True,
)

WORK.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(CHECKPOINT, "r") as archive:
    damaged = archive.testzip()

    if damaged is not None:
        raise RuntimeError(
            f"체크포인트 ZIP 손상 파일: {damaged}"
        )

    archive.extractall(WORK)

ML = WORK / "ml"
DATASET = ML / "dataset"

assert (ML / "import_real.py").exists(), (
    f"복원된 ml 폴더가 올바르지 않습니다: {ML}"
)

os.chdir(ML)

if str(ML) not in sys.path:
    sys.path.insert(0, str(ML))

print("\n복원된 ML 폴더:", ML)
print("복원된 DATASET:", DATASET)
print("상위 항목:", sorted(path.name for path in ML.iterdir())[:30])

In [ ]:
# 패키지 재설치
!pip install -q -r requirements.txt

from ultralytics import YOLO
from symbols import NAMES, KOREAN
from pathlib import Path
import shutil

IMAGE_SIZE = 800
BATCH = 8

DATA_YAML = ML / "data_colab.yaml"

# 합성 학습 모델을 우선 탐색
synthetic_candidates = [
    path
    for path in (ML / "runs").rglob("best.pt")
    if "evac_synthetic" in str(path)
]

if not synthetic_candidates:
    synthetic_candidates = list(
        (ML / "runs").rglob("best.pt")
    )

assert synthetic_candidates, (
    "복원한 체크포인트에서 합성 best.pt를 찾지 못했습니다."
)

BEST_SYN = max(
    synthetic_candidates,
    key=lambda path: path.stat().st_mtime,
)

print("복원된 BEST_SYN:", BEST_SYN)
print("파일 존재:", BEST_SYN.exists())
print(
    "크기:",
    round(BEST_SYN.stat().st_size / 1024 / 1024, 2),
    "MB",
)

# Drive에 별도 저장된 실제 라벨 ZIP 복원
drive_real_zip = (
    Path("/content/drive/MyDrive/evacuation_checkpoints")
    / "real_prelabeled_uploaded.zip"
)

runtime_real_zip = Path(
    "/content/real_prelabeled_uploaded.zip"
)

if drive_real_zip.exists():
    shutil.copy2(
        drive_real_zip,
        runtime_real_zip,
    )
    print("실제 라벨 ZIP 복원:", runtime_real_zip)
else:
    # ml 폴더 안에 업로드 ZIP이 들어 있는 경우 탐색
    real_zip_candidates = sorted(
        ML.glob("real_prelabeled*.zip"),
        key=lambda path: path.stat().st_mtime,
    )

    if real_zip_candidates:
        shutil.copy2(
            real_zip_candidates[-1],
            runtime_real_zip,
        )
        print(
            "ml 내부 ZIP 복원:",
            runtime_real_zip,
        )
    else:
        print(
            "실제 라벨 ZIP을 찾지 못했습니다. "
            "13번 셀에서 다시 업로드해야 합니다."
        )

print("\n복원 완료")

# label_assist.py가 결과를 image0.jpg 형식으로 반환하는 환경에서도
# 원래 입력 이미지 이름으로 라벨을 저장하도록 호환성 패치
assist_path = ML / "label_assist.py"
assist_text = assist_path.read_text(encoding="utf-8")
old_block = (
    "    for result in model.predict(**predict_kwargs):\n"
    "        result_count += 1\n"
    "        image_name = Path(result.path).name\n"
)
new_block = (
    "    for result_index, result in enumerate(model.predict(**predict_kwargs)):\n"
    "        result_count += 1\n"
    "        if result_index >= len(normalized):\n"
    "            raise RuntimeError('Model returned more results than input images')\n"
    "        image_name = normalized[result_index].name\n"
)
if old_block in assist_text:
    assist_path.write_text(
        assist_text.replace(old_block, new_block, 1),
        encoding="utf-8",
    )
    print("label_assist.py 이미지명 호환성 패치 완료")
elif "image_name = normalized[result_index].name" in assist_text:
    print("label_assist.py는 이미 패치되어 있습니다.")
else:
    print("주의: label_assist.py 자동 패치 위치를 찾지 못했습니다.")


In [ ]:
# 1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU가 없습니다. CPU로도 실행되지만 학습이 매우 느립니다."


## 2. 완성형 폴더 업로드

다운로드한 `ml_complete_200plus.zip`을 그대로 올리세요. ZIP 내부 최상위에는 `ml/` 폴더가 있습니다.
이미 업로드한 런타임에서 셀을 다시 실행하면 기존 폴더를 재사용합니다.


In [ ]:
import os, sys, zipfile, glob, shutil
from pathlib import Path

WORK = Path("/content/evac_work")
WORK.mkdir(parents=True, exist_ok=True)

def find_ml_dir(root: Path):
    hits = sorted(root.rglob("generate_dataset.py"))
    return hits[0].parent if hits else None

ML = find_ml_dir(WORK)
if ML is None:
    from google.colab import files
    print("ml_complete_200plus.zip을 선택하세요.")
    uploaded = files.upload()
    for name in uploaded:
        src = Path(name)
        if src.suffix.lower() == ".zip":
            with zipfile.ZipFile(src) as z:
                z.extractall(WORK)
        else:
            shutil.move(str(src), WORK / src.name)
    ML = find_ml_dir(WORK)

assert ML is not None, "generate_dataset.py를 찾지 못했습니다. ZIP 안에 ml/ 폴더가 있는지 확인하세요."
os.chdir(ML)
sys.path.insert(0, str(ML))
print("ML 폴더:", ML)
print("최상위 파일:", sorted(p.name for p in ML.iterdir()))


In [ ]:
# 3. 패키지 설치
!pip install -q -r requirements.txt
!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -f > /dev/null 2>&1

import ultralytics
print("ultralytics:", ultralytics.__version__)
ultralytics.checks()


In [ ]:
# 4. 데이터셋 확인 — 완성형 ZIP에는 500장이 들어 있습니다.
# 폴더를 가볍게 만들기 위해 dataset을 제거했다면 같은 시드로 다시 생성합니다.
DATASET = ML / "dataset"
EXPECTED = 500
current = sum(len(list((DATASET / "images" / s).glob("*.jpg"))) for s in ("train", "val", "test"))
if current < EXPECTED:
    print(f"현재 {current}장입니다. {EXPECTED}장까지 생성/재개합니다.")
    !python generate_dataset.py --count {EXPECTED} --out {DATASET} --preview 12

!python check_yolo_labels.py --dataset {DATASET} --report {DATASET}/label_check_report.json

for split in ("train", "val", "test"):
    images = list((DATASET / "images" / split).glob("*.jpg"))
    labels = list((DATASET / "labels" / split).glob("*.txt"))
    print(f"{split:>5}: images={len(images):>3} labels={len(labels):>3}")


In [ ]:
# 5. 합성 데이터 라벨 미리보기
from PIL import Image
import matplotlib.pyplot as plt

preview = DATASET / "preview_contact_sheet.jpg"
assert preview.exists(), preview
plt.figure(figsize=(20, 15))
plt.imshow(Image.open(preview))
plt.axis("off")
plt.show()


In [ ]:
# 6. Colab 절대경로용 data.yaml 생성
from symbols import NAMES, KOREAN

DATA_YAML = ML / "data_colab.yaml"
DATA_YAML.write_text(
    f"path: {DATASET}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n" + "".join(f"  {i}: {name}\n" for i, name in enumerate(NAMES)),
    encoding="utf-8",
)
print(DATA_YAML.read_text(encoding="utf-8"))


## 7. 1차 학습: 합성 정답 500장

좌우·상하 반전은 사용하지 않습니다. 글자와 대피 방향 화살표가 거울상이 되는 것을 막기 위해서입니다.
T4 메모리 부족이 발생하면 `BATCH = 4`로 낮추세요.


In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
EPOCHS = 140
IMAGE_SIZE = 800
BATCH = 8

model = YOLO(BASE_MODEL)
synthetic_run = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    patience=35,
    degrees=4.0,
    translate=0.06,
    scale=0.25,
    perspective=0.001,
    hsv_h=0.01,
    hsv_s=0.22,
    hsv_v=0.25,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.45,
    mixup=0.0,
    close_mosaic=15,
    project=str(ML / "runs"),
    name="evac_synthetic",
    seed=20260814,
)
BEST_SYN = Path(synthetic_run.save_dir) / "weights" / "best.pt"
print("1차 best.pt:", BEST_SYN)


In [ ]:
# 8. 합성 val/test 평가
model = YOLO(str(BEST_SYN))
metrics = model.val(data=str(DATA_YAML), imgsz=IMAGE_SIZE, split="val", verbose=False)
print(f"mAP50={float(metrics.box.map50):.4f}  mAP50-95={float(metrics.box.map):.4f}")
print(f"{'클래스':<15}{'정밀도':>9}{'재현율':>9}{'mAP50':>9}")
for i, cls_index in enumerate(metrics.box.ap_class_index):
    name = NAMES[int(cls_index)]
    print(f"{KOREAN[name]:<15}{float(metrics.box.p[i]):>9.3f}{float(metrics.box.r[i]):>9.3f}{float(metrics.box.ap50[i]):>9.3f}")


In [ ]:
# 9. 합성 test 이미지 예측 확인
TEST_IMAGES = sorted((DATASET / "images" / "test").glob("*.jpg"))
predictions = model.predict([str(p) for p in TEST_IMAGES[:4]], imgsz=IMAGE_SIZE, conf=0.30, verbose=False)
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
for ax, result in zip(axes.ravel(), predictions):
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(Path(result.path).name)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 10. 탐지 결과를 검수용 경로 그래프로 변환
TARGET = TEST_IMAGES[0]
PLAN_JSON = Path("/content/plan_draft.json")
PLAN_DEBUG = Path("/content/plan_debug.jpg")
!python predict_to_plan.py --model {BEST_SYN} --image {TARGET} --meters-wide 40 --out {PLAN_JSON} --debug-image {PLAN_DEBUG}
!node verify_plans.mjs {PLAN_JSON}

import json
plan = json.loads(PLAN_JSON.read_text(encoding="utf-8"))
print(plan["summary"])
plt.figure(figsize=(12, 9))
plt.imshow(Image.open(PLAN_DEBUG))
plt.axis("off")
plt.show()


# 2부. 실제 대피도 208장 밑라벨 생성

`real_data/photos_all/`에는 실제 대피도 원본 13개에서 만든 촬영환경 이미지 208장이 들어 있습니다.
이들은 **정답 라벨이 없는 입력 이미지**입니다. 먼저 1차 모델로 밑라벨을 만든 뒤, LabelImg·CVAT·Roboflow 등에서 모든 박스를 수정해야 합니다.

동일 원본의 16개 변형에는 같은 `srcNNN` 그룹이 붙어 있으며, 포함된 `split_manifest.json`이 원본 단위 train/val 분할을 유지합니다.


In [ ]:
# 11. 포함된 실제 이미지 208장에 밑라벨 생성
REAL_SOURCE = ML / "real_data" / "photos_all"
assert REAL_SOURCE.exists(), REAL_SOURCE
real_images = sorted(REAL_SOURCE.glob("*.jpg"))
print("실제 입력 이미지:", len(real_images))
assert len(real_images) >= 100, "실제 이미지가 100장 미만입니다. 완성형 ZIP인지 확인하세요."

PRELABEL = Path("/content/real_prelabeled")
shutil.rmtree(PRELABEL, ignore_errors=True)
!python label_assist.py --model {BEST_SYN} --images {REAL_SOURCE} --out {PRELABEL} --conf 0.18 --imgsz 960 --overwrite

# 원본 그룹 분할·출처 파일 보존
for src_name, dst_name in [
    ("split_manifest.json", "split_manifest.json"),
    ("metadata.csv", "source_metadata.csv"),
    ("ATTRIBUTION.md", "ATTRIBUTION.md"),
    ("sources.json", "sources.json"),
]:
    src = ML / "real_data" / src_name
    if src.exists():
        shutil.copy2(src, PRELABEL / dst_name)

shutil.make_archive("/content/real_prelabeled", "zip", PRELABEL)
from google.colab import files
files.download("/content/real_prelabeled.zip")


## 12. 로컬에서 밑라벨 수정

다운로드한 `real_prelabeled.zip`을 풀면 다음 파일이 있습니다.

```text
real_prelabeled/
├─ images/          # 실제 대피도 208장
├─ labels/          # 자동 밑라벨
├─ classes.txt      # 클래스 순서, 변경 금지
├─ REVIEW.md        # 우선 검수 목록
├─ split_manifest.json
└─ source_metadata.csv
```

검수 원칙:

1. `REVIEW.md` 위쪽 이미지부터 확인합니다.
2. 자동 모델이 놓친 비상구·계단·현재 위치는 라벨 파일에도 없으므로 직접 추가합니다.
3. 범례 아이콘이 실제 위치 객체로 잘못 라벨링되지 않았는지 확인합니다.
4. `room`은 방 영역, `door`는 문 개구부/스윙 영역으로 일관되게 표시합니다.
5. 파일명의 `srcNNN`과 `split_manifest.json`을 바꾸지 않습니다.
6. 검수 완료 후 폴더 전체를 다시 ZIP으로 묶습니다.
> 이 수정본은 `image0.txt` 형식으로 생성된 라벨도 `source_map.json` 순서로 안전하게 복구합니다.


In [ ]:
# ============================================================
# 13. real_prelabeled.zip 이름 복구 + 실제 데이터 병합
#    - 정상 same-stem ZIP과 image0.txt 형식의 버그 ZIP을 모두 처리
#    - GPU 불필요
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
import json
import re
import shutil
import subprocess
import sys
import zipfile

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp",
    ".bmp", ".tif", ".tiff",
}

ZIP_PATH = Path("/content/real_prelabeled_uploaded.zip")

if not ZIP_PATH.exists():
    print("검수한 real_prelabeled.zip을 선택하세요.")
    uploaded = files.upload()
    zip_name = next(
        (name for name in uploaded if name.lower().endswith(".zip")),
        None,
    )
    if zip_name is None:
        raise RuntimeError("ZIP 파일을 선택하지 않았습니다.")
    ZIP_PATH.write_bytes(uploaded[zip_name])

print("사용할 ZIP:", ZIP_PATH)
print("ZIP 크기 MB:", round(ZIP_PATH.stat().st_size / 1024 / 1024, 2))

SCAN_ROOT = Path("/content/real_pair_scan")
REPAIRED_ROOT = Path("/content/real_prelabeled_repaired")
shutil.rmtree(SCAN_ROOT, ignore_errors=True)
shutil.rmtree(REPAIRED_ROOT, ignore_errors=True)
SCAN_ROOT.mkdir(parents=True, exist_ok=True)
(REPAIRED_ROOT / "images").mkdir(parents=True, exist_ok=True)
(REPAIRED_ROOT / "labels").mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    damaged = archive.testzip()
    if damaged is not None:
        raise RuntimeError(f"ZIP 손상 파일: {damaged}")
    archive.extractall(SCAN_ROOT)

image_files = sorted(
    path.resolve()
    for path in SCAN_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS
    and "images" in {part.casefold() for part in path.parts}
)

label_files = sorted(
    path.resolve()
    for path in SCAN_ROOT.rglob("*.txt")
    if path.is_file()
    and "labels" in {part.casefold() for part in path.parts}
)

xml_files = sorted(
    path.resolve()
    for path in SCAN_ROOT.rglob("*.xml")
    if path.is_file()
)

print("\nZIP 진단")
print("이미지 수 :", len(image_files))
print("YOLO TXT  :", len(label_files))
print("XML       :", len(xml_files))
print("이미지 예시:", [p.name for p in image_files[:5]])
print("라벨 예시  :", [p.name for p in label_files[:5]])

if not image_files:
    raise RuntimeError("ZIP 안에서 images 폴더의 이미지를 찾지 못했습니다.")
if not label_files:
    raise RuntimeError("ZIP 안에서 labels 폴더의 YOLO TXT를 찾지 못했습니다.")

image_by_name = {}
for image_path in image_files:
    if image_path.name in image_by_name:
        raise RuntimeError(f"중복 이미지 파일명: {image_path.name}")
    image_by_name[image_path.name] = image_path

label_by_stem = {}
for label_path in label_files:
    if label_path.stem in label_by_stem:
        raise RuntimeError(f"중복 라벨 stem: {label_path.stem}")
    label_by_stem[label_path.stem] = label_path

# 1순위: 정상적인 image.jpg ↔ image.txt 구조
same_stem_pairs = [
    (image_path, label_by_stem.get(image_path.stem))
    for image_path in image_files
]

if all(label_path is not None for _, label_path in same_stem_pairs):
    pairs = [(image_path, label_path) for image_path, label_path in same_stem_pairs]
    mapping_mode = "same-stem"

else:
    # 2순위: 이번 프로젝트의 label_assist.py/Ultralytics 조합에서 발생한
    # image0.txt ... image207.txt 이름 버그를 source_map.json 순서로 복구
    source_map_hits = sorted(SCAN_ROOT.rglob("source_map.json"))
    if len(source_map_hits) != 1:
        raise RuntimeError(
            "이미지와 라벨 stem이 다르고 source_map.json도 정확히 1개가 아닙니다."
        )

    source_map = json.loads(
        source_map_hits[0].read_text(encoding="utf-8-sig")
    )
    if not isinstance(source_map, dict) or not source_map:
        raise RuntimeError("source_map.json 형식이 올바르지 않습니다.")

    source_names = list(source_map.keys())
    generic_labels = {}
    for label_path in label_files:
        match = re.fullmatch(r"image(\d+)", label_path.stem, flags=re.I)
        if match:
            generic_labels[int(match.group(1))] = label_path

    expected_indices = list(range(len(source_names)))
    if sorted(generic_labels) != expected_indices:
        raise RuntimeError(
            "imageN.txt 라벨 번호가 0부터 연속적이지 않습니다. "
            f"기대={len(source_names)}개, 발견={len(generic_labels)}개"
        )

    if set(source_names) != set(image_by_name):
        missing = sorted(set(source_names) - set(image_by_name))
        extra = sorted(set(image_by_name) - set(source_names))
        raise RuntimeError(
            "source_map.json과 images 폴더의 파일명이 다릅니다.\n"
            f"누락 예시={missing[:10]}\n추가 예시={extra[:10]}"
        )

    pairs = [
        (image_by_name[source_name], generic_labels[index])
        for index, source_name in enumerate(source_names)
    ]
    mapping_mode = "source_map insertion order: imageN -> N번째 원본 이미지"

print("\n매칭 방식:", mapping_mode)
print("연결된 이미지·라벨:", len(pairs))


def validate_yolo_label(path: Path) -> tuple[int, Counter]:
    box_count = 0
    class_counts = Counter()
    for line_no, raw in enumerate(
        path.read_text(encoding="utf-8-sig", errors="strict").splitlines(),
        1,
    ):
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(
                f"{path}:{line_no}: YOLO 라벨은 5개 열이어야 합니다."
            )
        cls_float = float(parts[0])
        cls_id = int(cls_float)
        coords = [float(value) for value in parts[1:]]
        if cls_float != cls_id or not 0 <= cls_id <= 7:
            raise ValueError(f"{path}:{line_no}: 잘못된 클래스 ID {parts[0]}")
        if not all(0.0 <= value <= 1.0 for value in coords):
            raise ValueError(f"{path}:{line_no}: 좌표가 0~1 범위를 벗어났습니다.")
        if coords[2] <= 0 or coords[3] <= 0:
            raise ValueError(f"{path}:{line_no}: width/height는 양수여야 합니다.")
        box_count += 1
        class_counts[cls_id] += 1
    return box_count, class_counts

empty_labels = []
total_boxes = 0
class_box_counts = Counter()

for image_path, label_path in pairs:
    boxes, counts = validate_yolo_label(label_path)
    if boxes == 0:
        empty_labels.append(label_path)
    total_boxes += boxes
    class_box_counts.update(counts)

    shutil.copy2(
        image_path,
        REPAIRED_ROOT / "images" / image_path.name,
    )
    shutil.copy2(
        label_path,
        REPAIRED_ROOT / "labels" / f"{image_path.stem}.txt",
    )

metadata_names = [
    "classes.txt",
    "split_manifest.json",
    "source_map.json",
    "source_metadata.csv",
    "metadata.csv",
    "ATTRIBUTION.md",
    "sources.json",
    "REVIEW.md",
    "review.json",
    "summary.json",
    "predictions.csv",
]
for filename in metadata_names:
    hits = sorted(SCAN_ROOT.rglob(filename))
    if hits:
        shutil.copy2(hits[0], REPAIRED_ROOT / filename)

repair_report = {
    "input_zip": str(ZIP_PATH),
    "mapping_mode": mapping_mode,
    "images": len(pairs),
    "labels": len(pairs),
    "empty_labels": len(empty_labels),
    "total_boxes": total_boxes,
    "class_box_counts": {
        str(class_id): class_box_counts[class_id]
        for class_id in range(8)
    },
}
(REPAIRED_ROOT / "repair_report.json").write_text(
    json.dumps(repair_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n라벨 검증")
print("빈 라벨 수 :", len(empty_labels))
print("전체 박스 수:", total_boxes)
print("클래스별 박스:", dict(sorted(class_box_counts.items())))
print("복구 폴더   :", REPAIRED_ROOT)

summary_hits = sorted(SCAN_ROOT.rglob("summary.json"))
if summary_hits:
    summary = json.loads(summary_hits[0].read_text(encoding="utf-8-sig"))
    if summary.get("detections") == total_boxes:
        print(
            "\n⚠️ 총 박스 수가 자동 밑라벨 summary.json과 정확히 같습니다. "
            "수동 검수 완료본인지 반드시 확인하세요."
        )

# 실제 데이터 병합
ML = Path(ML)
DATASET = Path(DATASET)
ALLOW_EMPTY_LABELS_FOR_PIPELINE_TEST = False

if empty_labels and not ALLOW_EMPTY_LABELS_FOR_PIPELINE_TEST:
    raise RuntimeError(
        f"빈 라벨이 {len(empty_labels)}개입니다. 사람이 확인한 뒤 다시 실행하세요."
    )

command = [
    sys.executable,
    "-u",
    str(ML / "import_real.py"),
    "--real",
    str(REPAIRED_ROOT),
    "--dataset",
    str(DATASET),
    "--clean-prefix",
]
if ALLOW_EMPTY_LABELS_FOR_PIPELINE_TEST:
    command.append("--allow-empty-label")

print("\n병합 명령:")
print(" ".join(command))

result = subprocess.run(
    command,
    cwd=str(ML),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("\n" + "=" * 70)
print("import_real.py 출력")
print("=" * 70)
print(result.stdout or "출력 없음")
print("=" * 70)
if result.returncode != 0:
    raise RuntimeError("import_real.py 실행에 실패했습니다.")

REAL_TRAIN_TXT = DATASET / "real_train.txt"
REAL_VAL_TXT = DATASET / "real_val.txt"


def read_nonempty_lines(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

train_paths = read_nonempty_lines(REAL_TRAIN_TXT)
val_paths = read_nonempty_lines(REAL_VAL_TXT)
all_paths = train_paths + val_paths
missing_images = [path for path in all_paths if not Path(path).exists()]
missing_labels = []
for image_name in all_paths:
    image_path = Path(image_name)
    label_path = Path(
        str(image_path).replace("/images/", "/labels/")
    ).with_suffix(".txt")
    if not label_path.exists():
        missing_labels.append(str(label_path))

print("\n최종 확인")
print("real_train.txt:", len(train_paths), "장")
print("real_val.txt  :", len(val_paths), "장")
print("누락 이미지   :", len(missing_images))
print("누락 라벨     :", len(missing_labels))

if not train_paths or not val_paths:
    raise RuntimeError("real_train.txt 또는 real_val.txt가 비어 있습니다.")
if missing_images or missing_labels:
    raise RuntimeError("목록에 기록된 이미지 또는 라벨 경로가 존재하지 않습니다.")

print("\n✅ 13번 실제 데이터 병합 성공")
print("다음 실행 순서: 14번 YAML 생성 → 15번 평가")


In [ ]:
# ============================================================
# 현재 ml 전체 상태를 Google Drive에 체크포인트로 저장
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import hashlib
import json
import datetime

ML = Path("/content/evac_work/ml")

assert ML.exists(), (
    f"{ML} 폴더가 없습니다. 현재 런타임이 이미 초기화되었는지 확인하세요."
)

# 학습 모델이 실제로 있는지 확인
best_candidates = sorted(
    (ML / "runs").rglob("best.pt"),
    key=lambda path: path.stat().st_mtime,
)

print("발견한 best.pt:")
for path in best_candidates:
    print(" -", path)

assert best_candidates, (
    "best.pt를 찾지 못했습니다. "
    "합성 학습이 완료되었는지 확인하세요."
)

# Google Drive 저장 폴더
DRIVE_DIR = Path(
    "/content/drive/MyDrive/evacuation_checkpoints"
)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# 시간별 체크포인트 이름
stamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

archive_base = Path(
    f"/content/evac_ml_checkpoint_{stamp}"
)

print("\nml 폴더를 압축합니다. 몇 분 걸릴 수 있습니다.")

archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(ML.parent),
        base_dir=ML.name,
    )
)

destination = DRIVE_DIR / archive_path.name

print("Google Drive로 복사 중...")
shutil.copy2(archive_path, destination)

# 별도로 존재할 수 있는 실제 라벨 ZIP도 보존
extra_zip_candidates = [
    Path("/content/real_prelabeled_uploaded.zip"),
    Path("/content/real_prelabeled.zip"),
]

for extra_zip in extra_zip_candidates:
    if extra_zip.exists():
        extra_destination = DRIVE_DIR / extra_zip.name
        shutil.copy2(extra_zip, extra_destination)
        print("추가 저장:", extra_destination)

# SHA-256 생성
sha256 = hashlib.sha256()

with destination.open("rb") as file:
    while True:
        chunk = file.read(1024 * 1024)

        if not chunk:
            break

        sha256.update(chunk)

digest = sha256.hexdigest()

sha_path = destination.with_suffix(
    destination.suffix + ".sha256"
)

sha_path.write_text(
    f"{digest}  {destination.name}\n",
    encoding="utf-8",
)

# 복원 참고 정보
checkpoint_info = {
    "created_at": stamp,
    "checkpoint": str(destination),
    "sha256": digest,
    "best_models": [
        str(path.relative_to(ML))
        for path in best_candidates
    ],
    "resume_from": "step_13_real_label_import",
}

info_path = DRIVE_DIR / (
    f"checkpoint_info_{stamp}.json"
)

info_path.write_text(
    json.dumps(
        checkpoint_info,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 70)
print("체크포인트 저장 완료")
print("=" * 70)
print("체크포인트:", destination)
print(
    "크기:",
    round(destination.stat().st_size / 1024 / 1024, 2),
    "MB",
)
print("SHA-256:", digest)
print("복원 정보:", info_path)
print("=" * 70)

In [ ]:
# 14. 합성+실제 학습용 YAML과 실제 검증 전용 YAML
MIXED_YAML = ML / "data_mixed.yaml"
REAL_YAML = ML / "data_real.yaml"
MIXED_YAML.write_text(
    f"path: {DATASET}\ntrain: images/train\nval: images/val\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)),
    encoding="utf-8",
)
REAL_YAML.write_text(
    f"path: {DATASET}\ntrain: real_train.txt\nval: real_val.txt\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)),
    encoding="utf-8",
)
print(REAL_YAML.read_text(encoding="utf-8"))


In [ ]:
# 15. 실제 검증 세트에서 미세조정 전 성능 기록
before = YOLO(str(BEST_SYN)).val(data=str(REAL_YAML), imgsz=IMAGE_SIZE, verbose=False)
print(f"미세조정 전 실제 mAP50={float(before.box.map50):.4f}  mAP50-95={float(before.box.map):.4f}")


In [ ]:
# 16. 합성+검수 실제 데이터로 미세조정
finetune = YOLO(str(BEST_SYN)).train(
    data=str(MIXED_YAML),
    epochs=80,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    lr0=0.0015,
    warmup_epochs=1.0,
    degrees=4.0,
    translate=0.05,
    scale=0.22,
    perspective=0.001,
    hsv_h=0.01,
    hsv_s=0.20,
    hsv_v=0.22,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.25,
    mixup=0.0,
    close_mosaic=10,
    project=str(ML / "runs"),
    name="evac_real_finetune",
    seed=20260814,
)
BEST_REAL = Path(finetune.save_dir) / "weights" / "best.pt"
print("최종 best.pt:", BEST_REAL)


In [ ]:
# 17. 실제 검증 전용 성능 비교
final_model = YOLO(str(BEST_REAL))
after = final_model.val(data=str(REAL_YAML), imgsz=IMAGE_SIZE, verbose=False)
print(f"실제 mAP50: {float(before.box.map50):.4f} -> {float(after.box.map50):.4f}")
print(f"{'클래스':<15}{'정밀도':>9}{'재현율':>9}{'mAP50':>9}")
for i, cls_index in enumerate(after.box.ap_class_index):
    name = NAMES[int(cls_index)]
    print(f"{KOREAN[name]:<15}{float(after.box.p[i]):>9.3f}{float(after.box.r[i]):>9.3f}{float(after.box.ap50[i]):>9.3f}")


In [ ]:
# 18. 최종 모델 ONNX 내보내기 및 다운로드
final_model.export(format="onnx", imgsz=IMAGE_SIZE, opset=12, simplify=True)

OUT = Path("/content/evac_model_final")
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir()
for source in [BEST_REAL, BEST_REAL.with_suffix(".onnx"), REAL_YAML, MIXED_YAML, DATASET / "real_import_report.json"]:
    if source.exists():
        shutil.copy2(source, OUT / source.name)
for name in ("results.png", "results.csv", "confusion_matrix_normalized.png"):
    source = Path(finetune.save_dir) / name
    if source.exists():
        shutil.copy2(source, OUT / name)
shutil.make_archive("/content/evac_model_final", "zip", OUT)
print("내보낸 파일:", sorted(p.name for p in OUT.iterdir()))
files.download("/content/evac_model_final.zip")


## 선택: 서로 다른 공개 원본을 더 수집

현재 포함된 208장은 실제 원본 13개의 촬영환경 변형입니다. 건물과 기호 체계의 다양성을 늘리려면 아래 명령으로 Wikimedia Commons 후보를 추가 수집한 뒤 `contact_sheet.jpg`를 검수하세요.

```python
!python crawl_wikimedia_evacuation.py --out /content/commons_evac --target 150 --depth 6 --clean
```

수집 결과도 정답 라벨이 아니므로 `label_assist.py` 실행 후 전량 수동 검수가 필요합니다.
